In [32]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [33]:
train = pd.read_csv('C:\\Users\\adaml\\Documents\\Stage\\Pricing engine\\Data\\raw\\train.csv')
train.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40


In [34]:
meal = pd.read_csv('C:\\Users\\adaml\\Documents\\Stage\\Pricing engine\\Data\\raw\\meal_info.csv')
meal.head()

,meal_id,category,cuisine
0,1885,Beverages,Thai
1,1993,Beverages,Thai
2,2539,Beverages,Thai
3,1248,Beverages,Indian
4,2631,Beverages,Indian


In [35]:
center = pd.read_csv('C:\\Users\\adaml\\Documents\\Stage\\Pricing engine\\Data\\raw\\fulfilment_center_info.csv')
center.head()

,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7
1,13,590,56,TYPE_B,6.7
2,124,590,56,TYPE_C,4.0
3,66,648,34,TYPE_A,4.1
4,94,632,34,TYPE_C,3.6


In [36]:
print(train.columns)
print(meal.columns)
print(center.columns)


Index(['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price',
       'emailer_for_promotion', 'homepage_featured', 'num_orders'],
      dtype='object')
Index(['meal_id', 'category', 'cuisine'], dtype='object')
Index(['center_id', 'city_code', 'region_code', 'center_type', 'op_area'], dtype='object')


In [37]:
data = pd.merge(train, meal, on='meal_id', how='left').merge(center, on='center_id', how='left')

In [38]:
data.columns

Index(['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price',
       'emailer_for_promotion', 'homepage_featured', 'num_orders', 'category',
       'cuisine', 'city_code', 'region_code', 'center_type', 'op_area'],
      dtype='object')

In [39]:
data.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


In [40]:
ID = data['id']
data.drop('id', axis=1, inplace=True)

In [41]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform the data
df = encoder.fit_transform(data[['center_type']])

# Convert back to a readable DataFrame with matching feature names
df_encoded = pd.DataFrame(df, columns=encoder.get_feature_names_out(['center_type']),index=data.index)

data = pd.concat([data, df_encoded], axis=1)

data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area,center_type_TYPE_A,center_type_TYPE_B,center_type_TYPE_C
0,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,0.0,0.0,1.0
1,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,0.0,0.0,1.0
2,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,0.0,0.0,1.0
3,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,0.0,0.0,1.0
4,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,0.0,0.0,1.0


In [42]:
data.drop(['center_type'], axis=1, inplace=True)
data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,op_area,center_type_TYPE_A,center_type_TYPE_B,center_type_TYPE_C
0,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,2.0,0.0,0.0,1.0
1,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,2.0,0.0,0.0,1.0
2,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,2.0,0.0,0.0,1.0
3,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,2.0,0.0,0.0,1.0
4,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,2.0,0.0,0.0,1.0


In [43]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Encode both columns
encoded = encoder.fit_transform(data[['cuisine', 'category']])

# Create DataFrame with generated column names
df_encoded = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['cuisine', 'category']),
    index=data.index
)

# Add encoded columns to original dataframe
data = pd.concat([data, df_encoded], axis=1)

data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,...,category_Fish,category_Other Snacks,category_Pasta,category_Pizza,category_Rice Bowl,category_Salad,category_Sandwich,category_Seafood,category_Soup,category_Starters
0,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
data.drop(['cuisine', 'category'], axis=1, inplace=True)
data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,...,category_Fish,category_Other Snacks,category_Pasta,category_Pizza,category_Rice Bowl,category_Salad,category_Sandwich,category_Seafood,category_Soup,category_Starters
0,1,55,1885,136.83,152.29,0,0,177,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,55,1993,136.83,135.83,0,0,270,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,55,2539,134.86,135.86,0,0,189,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,55,2139,339.50,437.53,0,0,54,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,55,2631,243.50,242.50,0,0,40,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
data.dtypes

week                       int64
center_id                  int64
meal_id                    int64
checkout_price           float64
base_price               float64
emailer_for_promotion      int64
homepage_featured          int64
num_orders                 int64
city_code                  int64
region_code                int64
op_area                  float64
center_type_TYPE_A       float64
center_type_TYPE_B       float64
center_type_TYPE_C       float64
cuisine_Continental      float64
cuisine_Indian           float64
cuisine_Italian          float64
cuisine_Thai             float64
category_Beverages       float64
category_Biryani         float64
category_Desert          float64
category_Extras          float64
category_Fish            float64
category_Other Snacks    float64
category_Pasta           float64
category_Pizza           float64
category_Rice Bowl       float64
category_Salad           float64
category_Sandwich        float64
category_Seafood         float64
category_S

In [47]:
data["revenue"] = data["num_orders"] * data["checkout_price"]

In [48]:
data.to_csv('C:\\Users\\adaml\\Documents\\Stage\\Pricing engine\\Data\\processed\\data.csv', index=False)